<a href="https://colab.research.google.com/github/samairaisabella-stack/Veritas-AI-Hackathon-2026/blob/main/Tvacha_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
%%bash
# Ensure the latest Google GenAI SDK and Gradio are installed
pip install -q google-genai gradio requests pillow

In [14]:
import gradio as gr
import requests
from PIL import Image
from google import genai
from google.colab import userdata
import traceback

def get_api_key():
    # Try Colab secrets
    try:
        return userdata.get('GEMINI_API_KEY')
    except Exception:
        try:
            return userdata.get('Gemini_API_Key')
        except Exception:
            return ""

def initial_skin_analysis(image):
    api_key = get_api_key()
    if not api_key:
        return "⚠️ No Gemini API Key found. Please add your key to Colab Secrets under `GEMINI_API_KEY` (or `Gemini_API_Key`) and ensure 'Notebook access' is enabled!", gr.update(visible=False), gr.update(visible=False)
    if image is None:
        return "⚠️ Please upload an image first.", gr.update(visible=False), gr.update(visible=False)

    try:
        client = genai.Client(api_key=api_key)
        prompt = """
        Analyze this skin image. Keep it brief:
        1. Describe the key visual characteristics (e.g., redness, dryness, rash, peeling, pigmentation).
        2. Determine if this skin state is sensitive to environmental or climate factors (such as sunlight/UV, dry air, heat, or pollution).
        3. Conclude with a clear YES or NO as to whether local climate factors should be cross-examined.
        """
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[image, prompt]
        )

        analysis_text = response.text

        if "yes" in analysis_text.lower() or "sun" in analysis_text.lower() or "uv" in analysis_text.lower() or "climate" in analysis_text.lower() or "weather" in analysis_text.lower():
            return (
                f"### 📋 Initial Skin Observations\n{analysis_text}\n\n💡 *This condition could be linked to environmental factors or UV exposure. Please enter your city below to analyze local climate risks.*",
                gr.update(visible=True),
                gr.update(visible=True)
            )
        else:
            return (
                f"### 📋 Initial Skin Observations\n{analysis_text}",
                gr.update(visible=False),
                gr.update(visible=False)
            )
    except Exception as e:
        err_stack = traceback.format_exc()
        return f"❌ Error during analysis:\n```\n{str(e)}\n```\n\n*Details for debugging:*\n```\n{err_stack}\n```", gr.update(visible=False), gr.update(visible=False)

def analyze_climate_correlation(city, image):
    api_key = get_api_key()
    if not api_key:
        return "⚠️ API key is missing. Add it to Colab Secrets."
    if not city:
        return "⚠️ Please enter a city name."

    try:
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1"
        geo_res = requests.get(geo_url, timeout=10).json()

        if "results" not in geo_res or len(geo_res["results"]) == 0:
            return f"❌ Could not find city: {city}. Please try again."

        top_result = geo_res["results"][0]
        lat = top_result["latitude"]
        lon = top_result["longitude"]
        resolved_name = f"{top_result['name']}, {top_result.get('country', '')}"

        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m,relative_humidity_2m,uv_index"
        w_data = requests.get(weather_url, timeout=10).json()["current"]

        air_url = f"https://air-quality-api.open-meteo.com/v1/air-quality?latitude={lat}&longitude={lon}&current=pm2_5"
        a_data = requests.get(air_url, timeout=10).json()["current"]

        climate_summary = (
            f"**Location:** {resolved_name} | "
            f"**Temp:** {w_data['temperature_2m']}°C | "
            f"**Humidity:** {w_data['relative_humidity_2m']}% | "
            f"**UV Index:** {w_data['uv_index']} | "
            f"**PM2.5:** {a_data['pm2_5']} µg/m³"
        )

        client = genai.Client(api_key=api_key)
        prompt = f"""
        You are a Climate & Dermatological Health AI Assistant.
        Analyze the attached skin image alongside these real-time local climate conditions:
        {climate_summary}

        Provide a structured, concise response with:
        1. Climate Impact Analysis: Explain how the current temperature, humidity, UV, or PM2.5 levels in {resolved_name} could worsen or trigger this condition.
        2. Actionable Protection Tips: 3 practical, non-prescription steps to protect this skin condition under these specific weather parameters today.
        3. End with a bold medical disclaimer stating this is educational AI analysis, not a medical diagnosis.
        """

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[image, prompt]
        )

        return f"### 🌍 Live Local Climate Data\n{climate_summary}\n\n---\n\n### 🔬 Climate & Skin Correlation Report\n{response.text}"

    except Exception as e:
        return f"❌ Correlation Error: {str(e)}"

# Dynamic Tab switching and showing helper
def get_started():
    return gr.update(visible=True), gr.update(selected=1)

# Build a customized Sage/Forest Green Theme
sage_theme = gr.themes.Soft(
    primary_hue=gr.themes.colors.emerald,
    secondary_hue=gr.themes.colors.stone,
    neutral_hue=gr.themes.colors.neutral,
).set(
    button_primary_background_fill="#2E4A3F",
    button_primary_background_fill_hover="#3D6354",
    button_primary_text_color="#FFFFFF",
    button_secondary_background_fill="#87A987",
    button_secondary_background_fill_hover="#9BB99B",
    button_secondary_text_color="#FFFFFF",
    background_fill_primary="#F7F9F6",
    block_background_fill="#FFFFFF",
    block_label_text_color="#2E4A3F",
    block_title_text_color="#2E4A3F"
)

with gr.Blocks() as demo:
    # Header brand
    gr.HTML("""
        <div style='text-align: center; padding: 15px;'>
            <h1 style='font-family: "Playfair Display", "Georgia", serif; font-weight: 700; color: #2E4A3F; font-size: 2.8em; margin: 0;'>T V A C H A &nbsp; A I</h1>
            <p style='font-family: "Montserrat", "Helvetica", sans-serif; text-align: center; color: #6D8276; font-size: 1.1em; letter-spacing: 1px; margin-bottom: 20px;'>INTELLIGENT DERMATOLOGY & ENVIRONMENTAL ANALYSIS</p>
        </div>
    """)

    # Navigation Control Tabs
    tabs = gr.Tabs()
    with tabs:
        # Tab 1: Front Welcome / Get Started
        with gr.TabItem("Welcome", id=0) as welcome_tab:
            gr.Markdown("""
            ### Welcome to Tvacha AI
            **Tvacha (त्वचा)** means *skin* in Sanskrit. This portal integrates clinical-grade visual AI with live global meteorological sensors to understand how your environment affects your skin.

            #### How it works:
            1. 📸 **Scan Skin:** Upload a photo of your skin concern for an instant initial visual analysis.
            2. 🌍 **Geological Mapping:** If environmental triggers (UV rays, extreme dryness, heat, or heavy pollution) are suspected, the application will prompt you for your current city.
            3. 🧬 **Environmental Correlation:** The system pulls live data from local weather and air-pollution monitoring stations to compile a tailored, real-time protection summary.
            """)

            get_started_btn = gr.Button("Get Started ➔", variant="primary")

        # Tab 2: The Core Workspace
        with gr.TabItem("Skin Analyzer Workspace", id=1, visible=False) as workspace_tab:
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### 🛠️ Config & Setup")
                    img_input = gr.Image(type="pil", label="📸 Upload Skin Photo")
                    analyze_btn = gr.Button("Scan Skin", variant="primary")

                    # Dynamic environmental input fields
                    city_input = gr.Textbox(
                        label="📍 Enter Your City to check local climate hazards",
                        placeholder="e.g., Delhi, California, London",
                        visible=False
                    )
                    climate_btn = gr.Button("Correlate with Live Local Weather", variant="secondary", visible=False)

                with gr.Column(scale=2):
                    gr.Markdown("### 📊 Live Diagnostic Panel")
                    output_box = gr.Markdown(value="*Ready. Upload an image and click 'Scan Skin' to start.*")

    # Link 'Get Started' button
    get_started_btn.click(
        fn=get_started,
        inputs=None,
        outputs=[workspace_tab, tabs]
    )

    # Action connections
    analyze_btn.click(
        fn=initial_skin_analysis,
        inputs=[img_input],
        outputs=[output_box, city_input, climate_btn]
    )

    climate_btn.click(
        fn=analyze_climate_correlation,
        inputs=[city_input, img_input],
        outputs=output_box
    )

demo.launch(share=True, theme=sage_theme, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://40246c5d84cef354a3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://40246c5d84cef354a3.gradio.live
